In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import astropy.units as u
import astropy.visualization
import named_arrays as na
import furst

In [ ]:
astropy.visualization.quantity_support();

In [ ]:
import warnings

warnings.filterwarnings("ignore", message="function 'sqrt' is not known")

In [ ]:
instrument = furst.instruments.design(num_wavelength=5)
system = instrument.system

axis_channel = "channel"
axis_wavelength = "wavelength"
axis_field = ("field_x", "field_y")
axis_pupil = ("pupil_x", "pupil_y")
axis_surface = system.axis_surface

In [ ]:
azimuth = instrument.feed_optic.rowland_azimuth
wavelength_min = instrument.wavelength_min.to(u.nm)
wavelength_max = instrument.wavelength_max.to(u.nm)

for i in range(azimuth.shape[axis_channel]):
    index = {axis_channel: i}
    print(
        f"channel {i}:"
        f" feed optic at {azimuth[index].ndarray:.2f},"
        f" {wavelength_min[index].ndarray:.1f} to {wavelength_max[index].ndarray:.1f}"
    )

In [ ]:
instrument_sparse = furst.instruments.design(
    num_wavelength=3,
    num_field=3,
    num_pupil=3,
)

fig, axs = plt.subplots(
    nrows=2,
    sharex=True,
    figsize=(9, 6),
    constrained_layout=True,
)
for ax, components in zip(axs, [("z", "x"), ("z", "y")]):
    instrument_sparse.system.plot(
        ax=ax,
        components=components,
        color="black",
        kwargs_rays=dict(
            color="tab:blue",
            linewidth=0.5,
        ),
    )
    ax.set_ylabel(f"${components[1]}$ ({ax.get_ylabel()})")
axs[1].set_xlabel(f"$z$ ({axs[1].get_xlabel()})")
axs[0].set_title("top view")
axs[1].set_title("side view")
axs[0].set_aspect("equal")

In [ ]:
raytrace = system.raytrace()
unvignetted = raytrace.outputs.unvignetted

index_surface = {
    "feed optic": 2,
    "grating": 3,
    "detector": 4,
}

fig, axs = plt.subplots(
    ncols=len(index_surface),
    figsize=(10, 4),
    constrained_layout=True,
)
for ax, name in zip(axs, index_surface):
    index = {axis_surface: index_surface[name]}
    surface = system.surfaces_all[index_surface[name]]
    position = surface.transformation.inverse(raytrace.outputs.position[index])
    na.plt.scatter(
        position.x,
        position.y,
        ax=ax,
        where=unvignetted[index],
        s=1,
    )
    wire = surface.aperture.wire()
    na.plt.plot(
        wire.x,
        wire.y,
        ax=ax,
        axis="wire",
        color="black",
    )
    ax.set_title(name)
    if name != "feed optic":
        ax.set_aspect("equal")

In [ ]:
axes_disk = axis_field + axis_pupil


def line_spread(instrument: furst.instruments.Instrument):
    """
    The positions of the rays on the detector, relative to the mean position
    of their wavelength, in pixels, along with the weight of each ray and
    the mean position of each wavelength.
    """
    rays = instrument.system.rayfunction_default.outputs
    weight = rays.unvignetted.astype(float)
    width_pixel = instrument.system.sensor.width_pixel
    position = rays.position
    position_mean = (position * weight).sum(axes_disk) / weight.sum(axes_disk)
    dx = (position.x - position_mean.x) / width_pixel
    dx = dx.to(u.dimensionless_unscaled) * u.pix
    return dx, weight, position_mean


width_pixel = system.sensor.width_pixel
position = system.rayfunction_default.outputs.position
dx, weight, position_mean = line_spread(instrument)

In [ ]:
index_center = {axis_wavelength: instrument.wavelength.shape[axis_wavelength] // 2}

lsf_2d = na.histogram2d(
    dx[index_center],
    position.y[index_center],
    bins=dict(lsf_x=21, lsf_y=11),
    axis=axes_disk,
    weights=weight[index_center],
    min=na.Cartesian2dVectorArray(-1 * u.pix, -8 * u.mm),
    max=na.Cartesian2dVectorArray(+1 * u.pix, +8 * u.mm),
)

fig, axs = na.plt.subplots(
    axis_cols=axis_channel,
    ncols=azimuth.shape[axis_channel],
    sharex=True,
    sharey=True,
    figsize=(10, 3.5),
    constrained_layout=True,
)
na.plt.pcolormesh(
    C=lsf_2d,
    ax=axs,
)
for i, ax in enumerate(axs.ndarray):
    ax.set_title(f"{azimuth[{axis_channel: i}].ndarray:.1f}")

In [ ]:
lsf = na.histogram(
    dx,
    bins=dict(lsf_x=41),
    axis=axes_disk,
    weights=weight,
    min=-1 * u.pix,
    max=+1 * u.pix,
)

fig, axs = na.plt.subplots(
    axis_cols=axis_channel,
    ncols=azimuth.shape[axis_channel],
    sharex=True,
    sharey=True,
    figsize=(10, 3),
    constrained_layout=True,
)
na.plt.stairs(
    lsf.inputs,
    lsf.outputs,
    ax=axs,
    axis="lsf_x",
)
for i, ax in enumerate(axs.ndarray):
    ax.set_title(f"{azimuth[{axis_channel: i}].ndarray:.1f}")
axs.ndarray[0].set_ylabel("rays");

In [ ]:
def width_lsf(dx: na.AbstractScalar, weight: na.AbstractScalar) -> na.AbstractScalar:
    """The standard deviation of the ray positions along the dispersion direction."""
    variance = (np.square(dx) * weight).sum(axes_disk) / weight.sum(axes_disk)
    return np.sqrt(variance)


wavelength = instrument.wavelength_physical.to(u.nm)
width = width_lsf(dx, weight)

fig, ax = plt.subplots(figsize=(8, 4), constrained_layout=True)
for i in range(azimuth.shape[axis_channel]):
    index = {axis_channel: i}
    na.plt.plot(
        wavelength[index],
        width[index],
        ax=ax,
        axis=axis_wavelength,
        marker="o",
        label=f"{azimuth[index].ndarray:.1f}",
    )
ax.set_xlabel(f"wavelength ({ax.get_xlabel()})")
ax.set_ylabel(f"LSF width ({ax.get_ylabel()})")
ax.legend(title="feed optic azimuth");

In [ ]:
def resolving_power_of(
    instrument: furst.instruments.Instrument,
    width: na.AbstractScalar,
    position_mean: na.AbstractCartesian3dVectorArray,
) -> na.AbstractScalar:
    """The resolving power of an instrument from the width of its line spread function."""
    wavelength = instrument.wavelength_physical
    width_pixel = instrument.system.sensor.width_pixel
    width_total = np.sqrt(np.square(width) + np.square(1 * u.pix) / 12)
    index_first = {axis_wavelength: 0}
    index_last = {axis_wavelength: ~0}
    dispersion = wavelength[index_last] - wavelength[index_first]
    dispersion = dispersion / (position_mean.x[index_last] - position_mean.x[index_first])
    wavelength_resolvable = 2 * width_total * (width_pixel / u.pix) * dispersion
    return (wavelength / wavelength_resolvable).to(u.dimensionless_unscaled)


resolving_power = resolving_power_of(instrument, width, position_mean)

fig, ax = plt.subplots(figsize=(8, 4), constrained_layout=True)
for i in range(azimuth.shape[axis_channel]):
    index = {axis_channel: i}
    na.plt.plot(
        wavelength[index],
        resolving_power[index],
        ax=ax,
        axis=axis_wavelength,
        marker="o",
        label=f"{azimuth[index].ndarray:.1f}",
    )
ax.set_xlabel(f"wavelength ({ax.get_xlabel()})")
ax.set_ylabel("resolving power")
ax.legend(title="feed optic azimuth");

In [ ]:
print(f"minimum resolving power: {resolving_power.min().ndarray:.0f}")
print(f"mean resolving power: {resolving_power.mean().ndarray:.0f}")
print(f"maximum resolving power: {resolving_power.max().ndarray:.0f}")

In [ ]:
fraction = unvignetted.mean(axes_disk + (axis_wavelength,))
names = [surface.name for surface in system.surfaces_all]

fig, ax = plt.subplots(figsize=(8, 4), constrained_layout=True)
for i in range(azimuth.shape[axis_channel]):
    index = {axis_channel: i}
    ax.plot(
        names,
        fraction[index].ndarray,
        marker="o",
        label=f"{azimuth[index].ndarray:.1f}",
    )
ax.set_ylim(bottom=0)
ax.set_ylabel("fraction of rays surviving")
ax.legend(title="feed optic azimuth");

In [ ]:
index_grating = {axis_surface: index_surface["grating"]}
index_detector = {axis_surface: index_surface["detector"]}

position_detector = system.sensor.transformation.inverse(
    raytrace.outputs.position[index_detector]
)

half_height_detector = instrument.camera.sensor.num_pixel_active.y * width_pixel / 2
half_height_detector = half_height_detector.to(u.mm)

distribution_y = na.histogram(
    position_detector.y,
    bins=dict(lsf_y=41),
    axis=axes_disk + (axis_wavelength,),
    weights=unvignetted[index_grating].astype(float),
    min=-20 * u.mm,
    max=+20 * u.mm,
)

fig, ax = plt.subplots(figsize=(8, 4), constrained_layout=True)
for i in range(azimuth.shape[axis_channel]):
    index = {axis_channel: i}
    na.plt.stairs(
        distribution_y.inputs,
        distribution_y.outputs[index],
        ax=ax,
        axis="lsf_y",
        label=f"{azimuth[index].ndarray:.1f}",
    )
ax.axvline(-half_height_detector, color="black", linestyle="--")
ax.axvline(+half_height_detector, color="black", linestyle="--")
ax.set_xlabel(f"height on the detector ({ax.get_xlabel()})")
ax.set_ylabel("rays")
ax.legend(title="feed optic azimuth");

In [ ]:
instrument_proposed = furst.instruments.design_proposed(num_wavelength=5)

designs = {
    "final": (instrument, "black"),
    "proposed": (instrument_proposed, "tab:red"),
}

fig, ax = plt.subplots(figsize=(9, 4), constrained_layout=True)
for label in designs:
    model, color = designs[label]
    model.system.plot(
        ax=ax,
        components=("z", "x"),
        plot_rays=False,
        color=color,
    )
    ax.plot([], [], color=color, label=label)
ax.set_xlabel(f"$z$ ({ax.get_xlabel()})")
ax.set_ylabel(f"$x$ ({ax.get_ylabel()})")
ax.set_aspect("equal")
ax.legend();

In [ ]:
origin = na.Cartesian3dVectorArray() * u.mm

components = {
    "feed optic": (instrument.feed_optic, instrument_proposed.feed_optic),
    "grating": (instrument.grating, instrument_proposed.grating),
    "detector": (instrument.camera.sensor, instrument_proposed.camera.sensor),
}

for name in components:
    final, proposed = components[name]
    displacement = final.transformation(origin) - proposed.transformation(origin)
    print(
        f"{name}:"
        f" dx from {displacement.x.min().ndarray:.3f} to {displacement.x.max().ndarray:.3f},"
        f" dz from {displacement.z.min().ndarray:.3f} to {displacement.z.max().ndarray:.3f}"
    )

In [ ]:
for label, model in [("final", instrument), ("proposed", instrument_proposed)]:
    wavelength_min = model.wavelength_min.to(u.nm)
    wavelength_max = model.wavelength_max.to(u.nm)
    print(
        f"{label}:"
        f" {wavelength_min.min().ndarray:.2f} to {wavelength_max.max().ndarray:.2f},"
        f" {(wavelength_max - wavelength_min).mean().ndarray:.3f} per channel"
    )

In [ ]:
dx_proposed, weight_proposed, position_mean_proposed = line_spread(instrument_proposed)
width_proposed = width_lsf(dx_proposed, weight_proposed)
resolving_power_proposed = resolving_power_of(
    instrument_proposed,
    width_proposed,
    position_mean_proposed,
)

for label, r in [("final", resolving_power), ("proposed", resolving_power_proposed)]:
    print(
        f"{label}:"
        f" minimum {r.min().ndarray:.0f},"
        f" mean {r.mean().ndarray:.0f},"
        f" maximum {r.max().ndarray:.0f}"
    )